In [11]:
import torch
import torchvision

from torchvision.transforms import PILToTensor
from torchvision.models.detection import (
    fasterrcnn_resnet50_fpn_v2,
    FasterRCNN_ResNet50_FPN_V2_Weights,
)
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torch.utils.data import Dataset
from torchvision import tv_tensors
from torchvision.transforms.v2 import functional as F

In [3]:
dataset = torchvision.datasets.Kitti(
    root="./data",
    transform=PILToTensor(),
    download=True
)

In [7]:
model = fasterrcnn_resnet50_fpn_v2(
    weights=FasterRCNN_ResNet50_FPN_V2_Weights.DEFAULT
)

in_features = model.roi_heads.box_predictor.cls_score.in_features
num_classes = 4

model.roi_heads.box_predictor = FastRCNNPredictor(
    in_features,
    num_classes
)

Downloading: "https://download.pytorch.org/models/fasterrcnn_resnet50_fpn_v2_coco-dd69338a.pth" to /Users/beckbargas/.cache/torch/hub/checkpoints/fasterrcnn_resnet50_fpn_v2_coco-dd69338a.pth


100%|██████████| 167M/167M [00:28<00:00, 6.16MB/s] 


In [ ]:
class KittiDetectionDataset(Dataset):
    def __init__(self, base_dataset, transforms=None):
        self.base_dataset = base_dataset
        self.transforms= transforms

        self.class_to_id = {
            "Car": 1,
            "Pedestrian": 2,
            "Cyclist": 3
        }

    def __len__(self):
        return len(self.base_dataset)

    def __getitem__(self, idx):
        image, target = self.base_dataset[idx]

        boxes = []
        labels = []

        for obj in target:
            class_name = obj["type"]

            if class_name not in self.class_to_id:
                continue

            boxes.append(obj["bbox"])
            labels.append(self.class_to_id[class_name])

        boxes = torch.tensor(boxes, dtype=torch.float32).reshape(-1, 4)
        labels = torch.tensor(labels, dtype=torch.int64)

        image = tv_tensors.Image(image)

        boxes = tv_tensors.BoundingBoxes(
            boxes, 
            format="XYXY",
            canvas_size=(F.getsize(image))
        )

        target = {
            "boxes": boxes,
            "labels": labels,
            "image_id": idx
        }

        if self.transforms is not None:
            image, target = self.transforms(image, target)

        return image, target